# Binh, Chapter 4 Homework
4.1.10a, 4.2.8, 4.3.6c, 4.4.4b, 4.4.12, 4.5.2b, 4.5.14, 4.7.2c, 4.7.4c

## 4.1.10a
Using the 6 provided points, we will be using the 5-point midpoint and endpoint estimation formulas. The first/last two points will use the latter, while the middle two points will use the former.

In [18]:
# setup chunk
import numpy as np
from matplotlib import pyplot as plt
import scipy

In [19]:


# where f is the array of points, i is the index we're calculating the derivative at and h is the step size
def five_pt_mid(f, i, h): 
    return (f[i-2] - 8*f[i-1] + 8*f[i+1] - f[i+2]) / (12*h)

# make function with 2 cases to handle left/right endpoints
def five_pt_end(f, i, h, forward=True):
    if forward:
        return (-25*f[i] + 48*f[i+1] - 36*f[i+2] + 16*f[i+3] - 3*f[i+4]) / (12*h)
    else:
        return (25*f[i] - 48*f[i-1] + 36*f[i-2] - 16*f[i-3] + 3*f[i-4]) / (12*h)

x = [1.05, 1.10, 1.15, 1.20, 1.25, 1.30]
fx = [-1.709847, -1.373823, -1.119214, -0.9160143, -0.7470223, -0.6015966]

# funnily enough, the actual formula only relies on f(x) and h
print("f'(1.05) =", five_pt_end(fx, 0, 0.05, True))
print("f'(1.10) =", five_pt_end(fx, 1, 0.05, True))
print("f'(1.15) =", five_pt_mid(fx, 2, 0.05))
print("f'(1.20) =", five_pt_mid(fx, 3, 0.05))
print("f'(1.25) =", five_pt_end(fx, 4, 0.05, False))
print("f'(1.30) =", five_pt_end(fx, 5, 0.05, False))

f'(1.05) = 7.798688499999975
f'(1.10) = 5.753751333333338
f'(1.15) = 4.499408166666668
f'(1.20) = 3.6755119999999986
f'(1.25) = 3.088419833333334
f'(1.30) = 2.7109926666666646


## 4.2.8
The answer to this exercise will be hand-written in a PDF. The file will be uploaded alongside the jupyter notebook.

## 4.3.6c
We are to numerically estimate:
$$
\int_{0.75}^{1.3} (\sin(x))^2-2x\sin(x) + 1 dx
$$
using Simpson's rule. We first declare the function in python, then compute the estimate using the provided formula.

In [20]:
x1 = (0.75+1.3)/2
h = x1 - 0.75

def f(x):
    return (np.sin(x))**2 - 2*x*np.sin(x) + 1

estimate = h/3 * (f(0.75) + 4*f(x1) + f(1.3))
print(estimate)

-0.020271589910295137


## 4.4.4b
We are asked to find
$$
\int_{-0.5}^{0.5}x \ln(x + 1)dx
$$
using the Composite Simpson's rule, with $n=6$. Using Algorithm 4.1, we will implement the estimation in Python:

In [21]:
def f(x):
    return x*np.log(x+1)

def comp_simp(f, a, b, n):
    h = (b-a)/n
    xi0 = f(a) + f(b)
    xi1 = 0
    xi2 = 0
    for i in range(1,n):
        x = a + i*h
        if i % 2 == 1:
            xi1 += f(x)
        else: xi2 += f(x)
    return h*(xi0 + 2*xi2 + 4*xi1)/3

comp_simp(f, -0.5, 0.5, 6)

np.float64(0.08809221096042885)

## 4.4.12
We are asked to estimate
$$
\int_{0}^{\pi} x^2 \cos(x) dx
$$
using three different techniques. Since the exercise doesn't seem too difficult to tackle by hand, we can solve for the analytical solution. After applying some elbow grease, we have
$$
\int_{0}^{\pi} x^2 \cos(x) dx = -2 \pi.
$$
We will print the value out in python, and make sure that all of our composite estimates are within $10^{-4}$ of the true answer.

In [22]:
soln = -2*np.pi

# declaring target function and relevant params
a = 0
b = np.pi
n = 400

def g(x):
    return x**2*np.cos(x)

# since composite simpson has been established, we can declare the other 2
def comp_trap(f, a, b, n):
    h = (b-a)/n
    fxj = 0
    for j in range(1, n):
        xj = a + j*h
        fxj += f(xj)
    return h/2*(f(a) + f(b) + 2*fxj)

def comp_mid(f,a,b,n):
    h = (b-a)/(n+2)
    total = 0 
    for j in range(0, n//2+1): # floor division to make sure an integer
        x2j = a + (2*j + 1) * h
        total += f(x2j)
    return 2*h*total

# using n=8 for all 3 methods, printing the results and the error
r1 = comp_trap(g,a,b,n)
r2 = comp_mid(g,a,b,n)
r3 = comp_simp(g,a,b,n)

print('Composite trapezoid error =', soln-r1)
print('Composite midpoint error =', soln-r2)
print('Composite Simpson error =', soln-r3)


Composite trapezoid error = 3.229830449491544e-05
Composite midpoint error = -6.395662537617142e-05
Composite Simpson error = -3.984652607869066e-10


It didn't take high n values to get the error for Composite Simpson to reach the required accuracy, but was the total reverse for the trapezoid and midpoint methods. With a high value of n, we did get all three estimates to within $10^{-4}$ of the actual value. 

## 4.5.2b
Using the code provided in the notebooks, we can implement the Romberg integration estimation technique as shown below:

In [23]:
# eliminating the specific entry, enabling more freedom
def romberg(f,a,b,k):
    R = np.zeros((k,k))
    for row in range(0,k):
        R[row][0] = comp_trap(f,a,b,2**row)
    for col in range(1,k):
        for row in range(col,k):
            R[row][col] = R[row][col-1]+(R[row][col-1]-R[row-1][col-1])/(4**(col)-1)
    return R

# still keep in mind that k is 0 indexed in python, but 1 indexed in the code.
# so r_33 in the text is actually r[2,2] using the formula
def h(x):
    return x*np.log(x+1)

A = romberg(h, -0.75, 0.75, 3)
print(A[2,2])

0.3279586101151939


## 4.5.14
According to the internet, there isn't a closed-form solution for the erf(x) function (though **incredibly** close numerical estimations are available through scipy). However, I will not make use of that for this exercise, but rather the absolute error between different diagonal entries. The python implementation is seen below:

In [24]:
def er(x):
    return (2/np.sqrt(np.pi)) * np.exp(-x**2)

B = romberg(er, 0, 1, 15)

for i in range(2,7):
    print("Absolute error between entry", i-1,
     "on the diagonal and the previous is", np.abs(B[i-1,i-1] - B[i-2,i-2]))

Absolute error between entry 1 on the diagonal and the previous is 0.07135949778492756
Absolute error between entry 2 on the diagonal and the previous is 0.00039123056386569655
Absolute error between entry 3 on the diagonal and the previous is 1.093553715447726e-05
Absolute error between entry 4 on the diagonal and the previous is 1.2932670934162616e-07
Absolute error between entry 5 on the diagonal and the previous is 3.191621411602341e-10


It took 5 diagonal entries to get the error under the desired bound of $10^{-7}$.

## 4.7.2c & 4.7.4c
Using u-substitution with $u=x^2-4$, we can pretty easily calculate the integral:
$$
\int_{3}^{3.5} \frac{x}{\sqrt{x^2-4}}dx = \sqrt{8.25} - \sqrt{5}
$$
We can then compute the actual solution and implement the Gaussian Quadrature formula in python to compute the error.

In [25]:
soln = np.sqrt(8.25) - np.sqrt(5)
a = 3
b = 3.5

def f7(x):
    return x/np.sqrt(x**2-4)
# now declare change of var function
def g7(s):
    return f7(((b-a)*s + a + b)/2)

# when n = 2
roots2, weights2 = scipy.special.roots_legendre(2)

def gauss2(f):
    total = 0
    for i in range(0,len(roots2)):
        total = total + weights2[i]*f(roots2[i])
    return total

# when n=3
roots3, weights3 = scipy.special.roots_legendre(3)

def gauss3(f):
    total = 0
    for i in range(0,len(roots3)):
        total = total + weights3[i]*f(roots3[i])
    return total

n2 = gauss2(g7)*(b-a)/2
n3 = gauss3(g7)*(b-a)/2
print("n=2 Estimate =", n2)
print('Error when n=2 =', np.abs(soln-n2))
print("n=3 Estimate =", n3)
print('Error when n=3 =', np.abs(soln-n3))

n=2 Estimate = 0.6361965649627966
Error when n=2 = 1.678080642786295e-05
n=3 Estimate = 0.6362131959130823
Error when n=3 = 1.4985614216200815e-07
